In [ ]:
import pandas as pd

In [ ]:
import tensorflow as tf
import pandas as pd
import numpy as np
from typing import Tuple, Dict

class ElasticityModel:
    def __init__(self, n_skus: int, n_features: int, ridge_lambda: float = 0.01):
        """
        Initialize the elasticity model
        
        Args:
            n_skus: Number of SKUs (407)
            n_features: Number of additional features (distribution, macro vars)
            ridge_lambda: Ridge regularization parameter
        """
        self.n_skus = n_skus
        self.n_features = n_features
        self.ridge_lambda = ridge_lambda
        
        # Initialize model parameters
        self._build_model()
        
    def _build_model(self):
        """Build the TensorFlow model architecture"""
        # Elasticity matrix: 407 x 407 (own + cross price elasticities)
        self.elasticity_matrix = tf.Variable(
            tf.random.normal([self.n_skus, self.n_skus], stddev=0.1),
            name='elasticity_matrix'
        )
        
        # Intercepts for each SKU
        self.intercepts = tf.Variable(
            tf.random.normal([self.n_skus], stddev=0.1),
            name='intercepts'
        )
        
        # Coefficients for additional features (distribution, macro)
        self.feature_coeffs = tf.Variable(
            tf.random.normal([self.n_skus, self.n_features], stddev=0.1),
            name='feature_coeffs'
        )
        
        # Optimizer
        self.optimizer = tf.optimizers.Adam(learning_rate=0.001)
        
    def predict(self, log_prices: tf.Tensor, features: tf.Tensor) -> tf.Tensor:
        """
        Predict log volumes
        
        Args:
            log_prices: [batch_size, n_skus] - log prices for all SKUs
            features: [batch_size, n_features] - additional features
            
        Returns:
            predicted_log_volumes: [batch_size, n_skus]
        """
        # Price elasticity effects: batch_size x n_skus x n_skus -> batch_size x n_skus
        price_effects = tf.reduce_sum(
            log_prices[:, None, :] * self.elasticity_matrix[None, :, :], 
            axis=2
        )
        
        # Feature effects: batch_size x n_skus
        feature_effects = tf.matmul(features, self.feature_coeffs, transpose_b=True)
        
        # Combined prediction
        predictions = (self.intercepts[None, :] + 
                      price_effects + 
                      feature_effects)
        
        return predictions
    
    def compute_loss(self, log_prices: tf.Tensor, features: tf.Tensor, 
                    true_log_volumes: tf.Tensor) -> Dict[str, tf.Tensor]:
        """
        Compute total loss with regularization and elasticity constraints
        
        Args:
            log_prices: [batch_size, n_skus]
            features: [batch_size, n_features] 
            true_log_volumes: [batch_size, n_skus]
            
        Returns:
            Dictionary of loss components
        """
        # Predictions
        predictions = self.predict(log_prices, features)
        
        # 1. Training Loss (MSE)
        mse_loss = tf.reduce_mean(tf.square(predictions - true_log_volumes))
        
        # 2. Ridge Regularization
        ridge_loss = self.ridge_lambda * (
            tf.reduce_sum(tf.square(self.elasticity_matrix)) +
            tf.reduce_sum(tf.square(self.feature_coeffs))
        )
        
        # 3. Elasticity Constraint Penalties
        constraint_loss = self._elasticity_constraint_loss()
        
        # Total loss
        total_loss = mse_loss + ridge_loss + constraint_loss
        
        return {
            'total_loss': total_loss,
            'mse_loss': mse_loss,
            'ridge_loss': ridge_loss,
            'constraint_loss': constraint_loss
        }
    
    def _elasticity_constraint_loss(self, penalty_weight: float = 1.0) -> tf.Tensor:
        """
        Compute penalty for violating elasticity bounds
        
        Returns:
            constraint_penalty: Scalar tensor
        """
        # Own-price elasticity constraints (diagonal should be negative, between -5 and 0)
        own_elasticities = tf.linalg.diag_part(self.elasticity_matrix)
        
        # Penalty for positive own-elasticities
        positive_own_penalty = tf.reduce_sum(
            tf.nn.relu(own_elasticities)  # Penalty when > 0
        )
        
        # Penalty for own-elasticities below -5
        extreme_negative_penalty = tf.reduce_sum(
            tf.nn.relu(-5.0 - own_elasticities)  # Penalty when < -5
        )
        
        # Cross-price elasticity constraints (off-diagonal should be positive, between 0 and 5)
        cross_elasticities = self.elasticity_matrix - tf.linalg.diag(own_elasticities)
        
        # Penalty for negative cross-elasticities
        negative_cross_penalty = tf.reduce_sum(
            tf.nn.relu(-cross_elasticities)  # Penalty when < 0
        )
        
        # Penalty for cross-elasticities above 5
        extreme_positive_penalty = tf.reduce_sum(
            tf.nn.relu(cross_elasticities - 5.0)  # Penalty when > 5
        )
        
        total_constraint_loss = penalty_weight * (
            positive_own_penalty + 
            extreme_negative_penalty + 
            negative_cross_penalty + 
            extreme_positive_penalty
        )
        
        return total_constraint_loss
    
    @tf.function
    def train_step(self, log_prices: tf.Tensor, features: tf.Tensor, 
                   true_log_volumes: tf.Tensor) -> Dict[str, tf.Tensor]:
        """Single training step"""
        with tf.GradientTape() as tape:
            losses = self.compute_loss(log_prices, features, true_log_volumes)
            total_loss = losses['total_loss']
        
        # Compute gradients
        gradients = tape.gradient(total_loss, [
            self.elasticity_matrix, 
            self.intercepts, 
            self.feature_coeffs
        ])
        
        # Apply gradients
        self.optimizer.apply_gradients(zip(gradients, [
            self.elasticity_matrix, 
            self.intercepts, 
            self.feature_coeffs
        ]))
        
        return losses
    
    def get_elasticity_matrix(self) -> np.ndarray:
        """Return the learned elasticity matrix"""
        return self.elasticity_matrix.numpy()
    
    def train(self, train_data: Dict, epochs: int = 1000, 
              validation_data: Dict = None) -> Dict:
        """
        Train the model
        
        Args:
            train_data: Dictionary with 'log_prices', 'features', 'log_volumes'
            epochs: Number of training epochs
            validation_data: Optional validation data
            
        Returns:
            Training history
        """
        history = {'train_loss': [], 'val_loss': []}
        
        for epoch in range(epochs):
            # Training step
            losses = self.train_step(
                train_data['log_prices'],
                train_data['features'], 
                train_data['log_volumes']
            )
            
            history['train_loss'].append(losses['total_loss'].numpy())
            
            # Validation
            if validation_data is not None:
                val_losses = self.compute_loss(
                    validation_data['log_prices'],
                    validation_data['features'],
                    validation_data['log_volumes']
                )
                history['val_loss'].append(val_losses['total_loss'].numpy())
            
            # Print progress
            if epoch % 100 == 0:
                print(f"Epoch {epoch}: Train Loss = {losses['total_loss']:.4f}")
                if validation_data is not None:
                    print(f"  Val Loss = {val_losses['total_loss']:.4f}")
        
        return history

# Usage Example
def create_and_train_model(data: pd.DataFrame, n_skus: int = 407):
    """
    Create and train the elasticity model
    
    Args:
        data: DataFrame with columns for prices, volumes, features
        n_skus: Number of SKUs
    """
    # Prepare data (placeholder - you'll need to implement data preprocessing)
    # log_prices = ... # Shape: [n_samples, n_skus]
    # features = ...   # Shape: [n_samples, n_features] 
    # log_volumes = ... # Shape: [n_samples, n_skus]
    
    # Initialize model
    model = ElasticityModel(n_skus=n_skus, n_features=5, ridge_lambda=0.01)
    
    # Train model
    # history = model.train(train_data, epochs=1000)
    
    # Get elasticity matrix
    # elasticity_matrix = model.get_elasticity_matrix()
    
    return model

In [ ]:
def create_sku_identifier(df):
    """Create standardized SKU identifier with spaces preserved"""
    # Handle missing values in SKU components first
    sku_columns = ['Brand', 'Sub-brand', 'Package', 'Package Type', 'Capacity Number']
    
    # Fill missing Sub-brand with empty string (will create double space, cleaned later)
    df['Sub-brand'] = df['Sub-brand'].fillna('')
    
    # Create SKU identifier with spaces
    df['sku'] = (df['Brand'].astype(str) + ' ' + 
                 df['Sub-brand'].astype(str) + ' ' + 
                 df['Package'].astype(str) + ' ' + 
                 df['Package Type'].astype(str) + ' ' + 
                 df['Capacity Number'].astype(str))
    
    # Clean only multiple consecutive spaces, preserve single spaces
    df['sku'] = df['sku'].str.replace(r'\s+', ' ', regex=True).str.strip()
    
    return df

def create_hierarchy_mapping(df, min_points=18):
    """
    Create mapping from original SKUs to effective SKUs with sufficient data
    
    Returns:
        - sku_mapping: Dict mapping original_sku -> effective_sku 
        - reverse_mapping: Dict mapping effective_sku -> list of original_skus
    """
    
    # Count observations per SKU
    sku_counts = df.groupby('sku').size()
    
    # Initialize mappings
    sku_mapping = {}
    reverse_mapping = {}
    
    # Define hierarchy levels
    def get_hierarchy_levels(row):
        """Generate hierarchy levels from most specific to most general"""
        brand = str(row['Brand'])
        subbrand = str(row['Sub-brand']) if pd.notna(row['Sub-brand']) and row['Sub-brand'] != '' else ''
        package = str(row['Package'])
        package_type = str(row['Package Type'])
        capacity = str(row['Capacity Number'])
        
        levels = [
            # Level 0: Full SKU (original)
            f"{brand} {subbrand} {package} {package_type} {capacity}".replace('  ', ' ').strip(),
            
            # Level 1: Brand + Package + Package Type + Capacity (drop sub-brand)
            f"{brand} {package} {package_type} {capacity}",
            
            # Level 2: Brand + Package Type + Capacity (drop package)
            f"{brand} {package_type} {capacity}",
            
            # Level 3: Brand + Package Type (drop capacity)
            f"{brand} {package_type}",
            
            # Level 4: Brand only
            f"{brand}"
        ]
        
        return levels
    
    # Process each unique SKU
    for sku in sku_counts.index:
        if sku_counts[sku] >= min_points:
            # SKU has sufficient data
            sku_mapping[sku] = sku
            if sku not in reverse_mapping:
                reverse_mapping[sku] = []
            reverse_mapping[sku].append(sku)
        else:
            # Find first hierarchy level with sufficient data
            sku_row = df[df['sku'] == sku].iloc[0]
            hierarchy_levels = get_hierarchy_levels(sku_row)
            
            effective_sku = None
            for level in hierarchy_levels[1:]:  # Skip level 0 (original SKU)
                # Count data points for this hierarchy level
                level_data = df[df.apply(lambda x: level in get_hierarchy_levels(x), axis=1)]
                if len(level_data) >= min_points:
                    effective_sku = level
                    break
            
            # Fallback to brand level if nothing else works
            if effective_sku is None:
                effective_sku = str(sku_row['Brand'])
            
            sku_mapping[sku] = effective_sku
            if effective_sku not in reverse_mapping:
                reverse_mapping[effective_sku] = []
            reverse_mapping[effective_sku].append(sku)
    
    return sku_mapping, reverse_mapping

def preprocess_with_hierarchy_collapse(sellout_df, ihs_df, min_points=18):
    """
    Complete preprocessing pipeline with hierarchical collapsing
    """
    
    # Step 1: Create original SKU identifier
    sellout_df = create_sku_identifier(sellout_df)
    
    # Step 2: Standard preprocessing
    sellout_df['Date'] = pd.to_datetime(sellout_df['Date'])
    ihs_df['Date'] = pd.to_datetime(ihs_df['Date'])
    
    # Remove missing volumes, handle other missing values
    sellout_df = sellout_df.dropna(subset=['Volume'])
    sellout_df = sellout_df.sort_values(['sku', 'Date'])
    sellout_df['Price'] = sellout_df.groupby('sku')['Price'].fillna(method='ffill')
    sellout_df['Weighted Distribution'] = sellout_df.groupby('sku')['Weighted Distribution'].fillna(method='ffill')
    
    # Step 3: Create hierarchy mapping
    print("Creating hierarchy mapping...")
    original_sku_mapping, reverse_mapping = create_hierarchy_mapping(sellout_df, min_points)
    
    # Step 4: Create effective dataset
    sellout_df['original_sku'] = sellout_df['sku']  # Store original SKU
    sellout_df['effective_sku'] = sellout_df['sku'].map(original_sku_mapping)
    
    # Aggregate data by effective SKU
    agg_df = sellout_df.groupby(['Date', 'effective_sku']).agg({
        'Volume': 'sum',
        'Price': 'mean',  # Average price within effective SKU
        'Weighted Distribution': 'mean',
        'original_sku': 'first'  # Keep reference to original SKU
    }).reset_index()
    
    # Step 5: Merge with macro data
    merged_df = pd.merge(agg_df, ihs_df, on='Date', how='left')
    macro_cols = [col for col in ihs_df.columns if col != 'Date']
    for col in macro_cols:
        merged_df[col] = merged_df[col].fillna(method='ffill')
    
    # Step 6: Log transformations
    merged_df['log_price'] = np.log(merged_df['Price'])
    merged_df['log_volume'] = np.log(merged_df['Volume'])
    merged_df['log_distribution'] = np.log(merged_df['Weighted Distribution'].clip(lower=0.01))
    
    return merged_df, original_sku_mapping, reverse_mapping

def prepare_model_data_with_mapping(df, original_sku_mapping, reverse_mapping):
    """
    Prepare model data ensuring elasticity mapping back to original SKUs
    """
    
    # Create matrices based on effective SKUs
    price_matrix = df.pivot(index='Date', columns='effective_sku', values='log_price')
    volume_matrix = df.pivot(index='Date', columns='effective_sku', values='log_volume')
    distribution_matrix = df.pivot(index='Date', columns='effective_sku', values='log_distribution')
    
    # Fill missing values
    price_matrix = price_matrix.fillna(method='ffill').fillna(method='bfill')
    volume_matrix = volume_matrix.fillna(method='ffill').fillna(method='bfill')
    distribution_matrix = distribution_matrix.fillna(method='ffill').fillna(method='bfill')
    
    # Create mapping from effective SKU index to original SKU names
    effective_skus = list(price_matrix.columns)
    n_effective_skus = len(effective_skus)
    
    return {
        'price_matrix': price_matrix,
        'volume_matrix': volume_matrix, 
        'distribution_matrix': distribution_matrix,
        'effective_skus': effective_skus,
        'n_effective_skus': n_effective_skus,
        'original_sku_mapping': original_sku_mapping,
        'reverse_mapping': reverse_mapping
    }

def expand_elasticity_matrix_to_original_skus(learned_elasticity_matrix, model_data_dict):
    """
    Map elasticity matrix from effective SKUs back to all original SKUs
    
    Args:
        learned_elasticity_matrix: [n_effective_skus, n_effective_skus] numpy array
        model_data_dict: Dictionary with mapping information
        
    Returns:
        original_elasticity_matrix: [n_original_skus, n_original_skus] with original SKU names
    """
    
    effective_skus = model_data_dict['effective_skus']
    reverse_mapping = model_data_dict['reverse_mapping']
    
    # Get all original SKUs
    all_original_skus = []
    for effective_sku in effective_skus:
        if effective_sku in reverse_mapping:
            all_original_skus.extend(reverse_mapping[effective_sku])
        else:
            all_original_skus.append(effective_sku)
    
    # Create expanded elasticity matrix
    n_original = len(all_original_skus)
    original_elasticity_matrix = np.zeros((n_original, n_original))
    
    # Map elasticities from effective to original SKUs
    for i, orig_sku_i in enumerate(all_original_skus):
        # Find which effective SKU this original SKU maps to
        effective_i = None
        for j, eff_sku in enumerate(effective_skus):
            if eff_sku in reverse_mapping and orig_sku_i in reverse_mapping[eff_sku]:
                effective_i = j
                break
            elif eff_sku == orig_sku_i:
                effective_i = j
                break
        
        for k, orig_sku_k in enumerate(all_original_skus):
            # Find which effective SKU this original SKU maps to
            effective_k = None
            for l, eff_sku in enumerate(effective_skus):
                if eff_sku in reverse_mapping and orig_sku_k in reverse_mapping[eff_sku]:
                    effective_k = l
                    break
                elif eff_sku == orig_sku_k:
                    effective_k = l
                    break
            
            # Copy elasticity from effective matrix
            if effective_i is not None and effective_k is not None:
                original_elasticity_matrix[i, k] = learned_elasticity_matrix[effective_i, effective_k]
    
    return original_elasticity_matrix, all_original_skus

# Updated model creation function
def create_and_train_model_with_hierarchy(data: pd.DataFrame):
    """
    Create and train elasticity model with hierarchical collapsing
    """
    
    # Load data
    sellout_train = pd.read_csv('/home/40107922/hackathon_2025/hackathon_2025/data/Sellout_Train.csv')
    ihs_data = pd.read_csv('/home/40107922/hackathon_2025/hackathon_2025/data/IHS_Masked.csv')
    
    # Preprocess with hierarchy collapse
    processed_df, sku_mapping, reverse_mapping = preprocess_with_hierarchy_collapse(
        sellout_train, ihs_data, min_points=18
    )
    
    # Prepare model data
    model_data = prepare_model_data_with_mapping(processed_df, sku_mapping, reverse_mapping)
    
    print(f"Original SKUs: {len(sku_mapping)}")
    print(f"Effective SKUs for modeling: {model_data['n_effective_skus']}")
    
    # Initialize model with effective SKU count
    model = ElasticityModel(
        n_skus=model_data['n_effective_skus'], 
        n_features=5, 
        ridge_lambda=0.01
    )
    
    # Prepare training data
    train_data = {
        'log_prices': tf.constant(model_data['price_matrix'].values, dtype=tf.float32),
        'features': tf.constant(model_data['distribution_matrix'].values, dtype=tf.float32),  # Simplified
        'log_volumes': tf.constant(model_data['volume_matrix'].values, dtype=tf.float32)
    }
    
    # Train model
    history = model.train(train_data, epochs=1000)
    
    # Get elasticity matrix for effective SKUs
    effective_elasticity_matrix = model.get_elasticity_matrix()
    
    # Expand back to original SKUs
    original_elasticity_matrix, original_sku_names = expand_elasticity_matrix_to_original_skus(
        effective_elasticity_matrix, model_data
    )
    
    return {
        'model': model,
        'history': history,
        'effective_elasticity_matrix': effective_elasticity_matrix,
        'original_elasticity_matrix': original_elasticity_matrix,
        'original_sku_names': original_sku_names,
        'effective_sku_names': model_data['effective_skus'],
        'sku_mapping': sku_mapping,
        'reverse_mapping': reverse_mapping
    }